# Directory Management and Template Update Notebook

This notebook will:
1. Create a directory listing of template files
2. Save the listing to a Markdown file
3. Update base.html with the new listing
4. Update server.py to handle any necessary changes

This automation helps maintain consistency between the file system and application code.

## Import Required Libraries

In [ ]:
import os
import re
import datetime
import shutil
from pathlib import Path

# Define color for status messages
class Colors:
    SUCCESS = '\033[92m'  # Green
    WARNING = '\033[93m'  # Yellow
    ERROR = '\033[91m'    # Red
    ENDC = '\033[0m'      # Reset color

def print_status(message, status_type="info"):
    if status_type == "success":
        print(f"{Colors.SUCCESS}[SUCCESS]{Colors.ENDC} {message}")
    elif status_type == "warning":
        print(f"{Colors.WARNING}[WARNING]{Colors.ENDC} {message}")
    elif status_type == "error":
        print(f"{Colors.ERROR}[ERROR]{Colors.ENDC} {message}")
    else:
        print(f"[INFO] {message}")

## List Files in Directory

We'll scan the 'src/web/templates' directory and create a structured listing of all template files.

In [ ]:
def list_templates_directory(base_path="src/web/templates"):
    """
    Create a structured listing of all files in the templates directory
    
    Args:
        base_path: Path to the templates directory
        
    Returns:
        dict: Dictionary with template information structured by category
    """
    # Check if the directory exists
    if not os.path.exists(base_path):
        print_status(f"Directory {base_path} does not exist", "error")
        return None
    
    templates = {}
    
    # Walk through directory tree
    for root, dirs, files in os.walk(base_path):
        # Skip hidden directories
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        
        # Get relative path
        rel_path = os.path.relpath(root, base_path)
        if rel_path == '.':
            category = 'root'
        else:
            category = rel_path
            
        # List only HTML files
        html_files = [f for f in files if f.endswith('.html')]
        
        if html_files:
            if category not in templates:
                templates[category] = []
                
            for file in html_files:
                file_path = os.path.join(root, file)
                # Get file metadata
                templates[category].append({
                    'name': file,
                    'path': os.path.join(rel_path, file) if rel_path != '.' else file,
                    'last_modified': datetime.datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S'),
                    'size': os.path.getsize(file_path)
                })
    
    return templates

# Test the function
templates_data = list_templates_directory()

if templates_data:
    print_status(f"Found {sum(len(files) for files in templates_data.values())} template files", "success")
    for category, files in templates_data.items():
        print(f"\n{category.upper()} ({len(files)} files):")
        for file in files:
            print(f"  - {file['name']} ({file['path']})")
else:
    print_status("No template files found or directory doesn't exist", "warning")

## Write Directory Listing to Markdown File

Now we'll write the directory listing to a Markdown file for documentation and potential use in the application.

In [ ]:
def write_templates_to_markdown(templates_data, output_file="templates_listing.md"):
    """
    Write the template listing to a markdown file
    
    Args:
        templates_data: Dictionary with template information
        output_file: Path to the output markdown file
        
    Returns:
        bool: True if successful, False otherwise
    """
    if not templates_data:
        print_status("No template data to write", "error")
        return False
    
    try:
        with open(output_file, 'w') as f:
            f.write("# Template Files Directory\n\n")
            f.write(f"*Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")
            
            total_files = sum(len(files) for files in templates_data.values())
            f.write(f"**Total files: {total_files}**\n\n")
            
            for category, files in templates_data.items():
                f.write(f"## {category.upper()}\n\n")
                
                # Create table header
                f.write("| File Name | Path | Last Modified | Size (bytes) |\n")
                f.write("|-----------|------|--------------|-------------:|\n")
                
                for file in files:
                    f.write(f"| {file['name']} | {file['path']} | {file['last_modified']} | {file['size']} |\n")
                
                f.write("\n")
        
        print_status(f"Successfully wrote template listing to {output_file}", "success")
        return True
    
    except Exception as e:
        print_status(f"Error writing markdown file: {str(e)}", "error")
        return False

# Write the templates data to a markdown file
if templates_data:
    success = write_templates_to_markdown(templates_data)
    if success:
        print_status("Template listing has been saved to templates_listing.md", "success")
else:
    print_status("No template data to write", "error")

## Update base.html

Now we'll update the base.html file to include a navigation menu based on the template listing.
We'll read the base.html file, find the navigation section, and update it with links to all templates.

In [ ]:
def update_base_html(templates_data, base_html_path="src/web/templates/base.html", backup=True):
    """
    Update the base.html file with links to all templates
    
    Args:
        templates_data: Dictionary with template information
        base_html_path: Path to the base.html file
        backup: Whether to create a backup of the original file
        
    Returns:
        bool: True if successful, False otherwise
    """
    if not templates_data:
        print_status("No template data to update base.html with", "error")
        return False
    
    if not os.path.exists(base_html_path):
        print_status(f"Base HTML file {base_html_path} does not exist", "error")
        return False
    
    try:
        # Create backup if requested
        if backup:
            backup_path = f"{base_html_path}.bak.{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"
            shutil.copy2(base_html_path, backup_path)
            print_status(f"Created backup at {backup_path}", "success")
        
        # Read the original file
        with open(base_html_path, 'r') as f:
            content = f.read()
        
        # Define patterns for navigation section
        nav_section_pattern = r'<!-- BEGIN TEMPLATE NAVIGATION -->(.*?)<!-- END TEMPLATE NAVIGATION -->'
        nav_section_match = re.search(nav_section_pattern, content, re.DOTALL)
        
        if not nav_section_match:
            # If the navigation section doesn't exist, we'll insert it before </nav>
            new_nav_section = """
            <!-- BEGIN TEMPLATE NAVIGATION -->
            <ul class="navbar-nav template-nav">
                <!-- Template navigation will be inserted here -->
            </ul>
            <!-- END TEMPLATE NAVIGATION -->
            """
            content = content.replace('</nav>', f'{new_nav_section}\n</nav>')
        
        # Build the new navigation HTML
        new_nav_html = "\n            <!-- BEGIN TEMPLATE NAVIGATION -->\n"
        new_nav_html += "            <ul class=\"navbar-nav template-nav\">\n"
        
        # Add entries for each category
        for category, files in templates_data.items():
            if category == 'root':
                category_display = 'Root Templates'
            else:
                category_display = category.capitalize()
                
            new_nav_html += f"                <li class=\"nav-item dropdown\">\n"
            new_nav_html += f"                    <a class=\"nav-link dropdown-toggle\" href=\"#\" role=\"button\" data-bs-toggle=\"dropdown\">{category_display}</a>\n"
            new_nav_html += f"                    <ul class=\"dropdown-menu\">\n"
            
            for file in files:
                # Skip base.html itself to avoid confusion
                if file['name'] == 'base.html':
                    continue
                    
                # Create path for link
                link_path = f"/{file['path']}" if not file['path'].startswith('/') else file['path']
                new_nav_html += f"                        <li><a class=\"dropdown-item\" href=\"{link_path}\">{file['name']}</a></li>\n"
                
            new_nav_html += f"                    </ul>\n"
            new_nav_html += f"                </li>\n"
        
        new_nav_html += "            </ul>\n            <!-- END TEMPLATE NAVIGATION -->"
        
        # Replace the navigation section
        if nav_section_match:
            updated_content = re.sub(nav_section_pattern, new_nav_html, content, flags=re.DOTALL)
        else:
            # This is a fallback if we couldn't find the section markers but added them
            updated_content = content
        
        # Write the updated file
        with open(base_html_path, 'w') as f:
            f.write(updated_content)
        
        print_status(f"Successfully updated navigation in {base_html_path}", "success")
        return True
        
    except Exception as e:
        print_status(f"Error updating base.html: {str(e)}", "error")
        return False

# Update base.html with the template navigation
if templates_data:
    success = update_base_html(templates_data)
    if success:
        print_status("base.html has been updated with template navigation", "success")
else:
    print_status("No template data to update base.html with", "error")

## Update server.py

Finally, we'll update server.py to ensure it can correctly serve all the template files and handle any necessary routing based on the template listing.

In [ ]:
def update_server_py(templates_data, server_py_path="src/web/server.py", backup=True):
    """
    Update server.py to handle routing for all templates
    
    Args:
        templates_data: Dictionary with template information
        server_py_path: Path to the server.py file
        backup: Whether to create a backup of the original file
        
    Returns:
        bool: True if successful, False otherwise
    """
    if not templates_data:
        print_status("No template data to update server.py with", "error")
        return False
    
    if not os.path.exists(server_py_path):
        print_status(f"Server Python file {server_py_path} does not exist", "error")
        return False
    
    try:
        # Create backup if requested
        if backup:
            backup_path = f"{server_py_path}.bak.{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"
            shutil.copy2(server_py_path, backup_path)
            print_status(f"Created backup at {backup_path}", "success")
        
        # Read the original file
        with open(server_py_path, 'r') as f:
            content = f.readlines()
        
        # Look for the section where routes are defined
        routes_section_start = None
        routes_section_end = None
        has_template_routes_marker = False
        
        for i, line in enumerate(content):
            if '# BEGIN TEMPLATE ROUTES' in line:
                routes_section_start = i
                has_template_routes_marker = True
            elif '# END TEMPLATE ROUTES' in line and routes_section_start is not None:
                routes_section_end = i
                break
        
        # If no explicit section markers, look for app = Flask(__name__) to insert after
        if routes_section_start is None:
            for i, line in enumerate(content):
                if 'app = Flask' in line:
                    # Insert a few lines after Flask app initialization
                    routes_section_start = i + 3
                    routes_section_end = routes_section_start
                    break
        
        # If we still couldn't find where to insert routes, just append to the end
        if routes_section_start is None:
            routes_section_start = len(content) - 1
            routes_section_end = routes_section_start
        
        # Create route handlers for each template
        new_routes = []
        if not has_template_routes_marker:
            new_routes.append("\n# BEGIN TEMPLATE ROUTES\n")
        
        # Add a comment explaining these are auto-generated
        new_routes.append("# Auto-generated template routes - DO NOT EDIT\n")
        new_routes.append("# Generated on: " + datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n\n")
        
        # First, create a function that handles template rendering in a generic way
        new_routes.append("def render_template_with_defaults(template_path, **kwargs):\n")
        new_routes.append("    \"\"\"Render a template with default context variables\"\"\"\n")
        new_routes.append("    # Add any default context variables here\n")
        new_routes.append("    context = {\n")
        new_routes.append("        'app_name': 'Impression Core',\n")
        new_routes.append("        'current_year': datetime.datetime.now().year\n")
        new_routes.append("    }\n")
        new_routes.append("    # Update with any passed kwargs\n")
        new_routes.append("    context.update(kwargs)\n")
        new_routes.append("    return render_template(template_path, **context)\n\n")
        
        # Create a route for template listing
        new_routes.append("@app.route('/templates')\n")
        new_routes.append("def template_listing():\n")
        new_routes.append("    \"\"\"Display a list of all available templates\"\"\"\n")
        new_routes.append("    return render_template_with_defaults('templates_index.html', templates={\n")
        
        for category, files in templates_data.items():
            category_display = category if category != 'root' else 'Root Templates'
            new_routes.append(f"        '{category_display}': [\n")
            
            for file in files:
                # Skip base.html from the listing
                if file['name'] == 'base.html':
                    continue
                    
                route_path = "/" + file['path'].replace('\\', '/') if category != 'root' else "/" + file['name']
                new_routes.append(f"            {{'name': '{file['name']}', 'path': '{route_path}', 'last_modified': '{file['last_modified']}'}},\n")
                
            new_routes.append("        ],\n")
            
        new_routes.append("    })\n\n")
        
        # Now create routes for each template file
        for category, files in templates_data.items():
            for file in files:
                # Skip base.html as it's not meant to be rendered directly
                if file['name'] == 'base.html':
                    continue
                    
                template_path = file['path']
                route_path = "/" + template_path.replace('\\', '/') if category != 'root' else "/" + file['name']
                
                # Remove .html extension from route if it exists
                if route_path.endswith('.html'):
                    route_url = route_path[:-5]
                else:
                    route_url = route_path
                
                # Create the route
                new_routes.append(f"@app.route('{route_url}')\n")
                new_routes.append(f"def serve_{category.replace('/', '_').replace('.', '_')}_{file['name'].replace('.', '_').replace('-', '_')}():\n")
                new_routes.append(f"    \"\"\"Serve the {file['name']} template\"\"\"\n")
                new_routes.append(f"    return render_template_with_defaults('{template_path}')\n\n")
                
        if not has_template_routes_marker:
            new_routes.append("# END TEMPLATE ROUTES\n")
        
        # Insert or replace the routes section
        new_content = content[:routes_section_start] + new_routes + content[routes_section_end+1:] if has_template_routes_marker else content[:routes_section_start] + new_routes + content[routes_section_start:]
        
        # Check if we need to import datetime
        has_datetime_import = False
        for line in new_content:
            if 'import datetime' in line:
                has_datetime_import = True
                break
                
        if not has_datetime_import:
            # Find the imports section
            for i, line in enumerate(new_content):
                if 'from flask import' in line:
                    new_content.insert(i+1, "import datetime\n")
                    break
        
        # Write the updated file
        with open(server_py_path, 'w') as f:
            f.writelines(new_content)
        
        print_status(f"Successfully updated routes in {server_py_path}", "success")
        return True
        
    except Exception as e:
        print_status(f"Error updating server.py: {str(e)}", "error")
        return False

# Update server.py with routes for all templates
if templates_data:
    success = update_server_py(templates_data)
    if success:
        print_status("server.py has been updated with routes for all templates", "success")
else:
    print_status("No template data to update server.py with", "error")

## Summary and Verification

Let's summarize what we've accomplished and verify that all the changes have been made correctly.

In [ ]:
def verify_changes():
    """
    Verify that all changes have been made correctly
    """
    verification_results = {
        "templates_listing.md": os.path.exists("templates_listing.md"),
        "base.html": False,
        "server.py": False
    }
    
    # Check if base.html was modified
    if os.path.exists("src/web/templates/base.html"):
        with open("src/web/templates/base.html", 'r') as f:
            content = f.read()
            verification_results["base.html"] = "<!-- BEGIN TEMPLATE NAVIGATION -->" in content
            
    # Check if server.py was modified
    if os.path.exists("src/web/server.py"):
        with open("src/web/server.py", 'r') as f:
            content = f.read()
            verification_results["server.py"] = "# BEGIN TEMPLATE ROUTES" in content or "# Auto-generated template routes" in content

    # Print verification results
    print("Verification Results:")
    for item, success in verification_results.items():
        if success:
            print(f"✅ {item}: Changes verified")
        else:
            print(f"❌ {item}: Changes not verified or file not found")
            
    # Overall success
    if all(verification_results.values()):
        print_status("All changes have been successfully applied!", "success")
    else:
        print_status("Some changes could not be verified. Please check the logs above.", "warning")

# Run verification
verify_changes()

print("\nSummary:")
print("1. Created a listing of all template files")
print("2. Saved the listing to templates_listing.md")
print("3. Updated base.html with navigation links to all templates")
print("4. Updated server.py with routes to handle all templates")
print("\nNext steps:")
print("1. Verify the application still works correctly")
print("2. Test the new navigation links")
print("3. Check that all templates are accessible via their routes")

# Directory Management Notebook

This notebook provides utilities to create a directory listing, save it to Markdown format, and use it to update relevant files in a web application such as base.html and server.py.

## Import Required Libraries

We'll import the necessary libraries for directory operations, file handling, and other utilities.

In [ ]:
# Import required libraries
import os
import datetime
import re
import shutil
from pathlib import Path

## List Files in Directory

Using os.listdir() to enumerate all files in the 'src/web/templates' directory. This section will:
1. List all files in the directory
2. Filter for specific file types if needed
3. Create a structured representation of the directory contents

In [ ]:
# Define the target directory path
template_dir = os.path.join("src", "web", "templates")

# Ensure the directory exists
if not os.path.exists(template_dir):
    print(f"Error: Directory {template_dir} does not exist. Creating it...")
    os.makedirs(template_dir, exist_ok=True)

# List all files in the directory
template_files = []
try:
    template_files = os.listdir(template_dir)
    print(f"Found {len(template_files)} files in {template_dir}")
except Exception as e:
    print(f"Error listing directory: {e}")

# Filter for HTML files if needed
html_files = [f for f in template_files if f.endswith('.html')]
print(f"Found {len(html_files)} HTML files")

# Create a structured representation of the directory
directory_structure = {
    "path": template_dir,
    "all_files": template_files,
    "html_files": html_files,
    "date_generated": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

directory_structure

## Write Directory Listing to Markdown File

This section will:
1. Format the directory listing into a well-structured Markdown document
2. Save the output to a file named templates_listing.md

In [ ]:
# Format the directory listing as Markdown
def create_markdown_listing(dir_structure):
    md_content = f"# Templates Directory Listing\n\n"
    md_content += f"Generated on: {dir_structure['date_generated']}\n\n"
    md_content += f"Directory: `{dir_structure['path']}`\n\n"
    
    # Add HTML files section
    md_content += "## HTML Files\n\n"
    if dir_structure['html_files']:
        md_content += "| Filename | Last Modified | Size |\n"
        md_content += "|----------|--------------|------|\n"
        
        for file in dir_structure['html_files']:
            file_path = os.path.join(dir_structure['path'], file)
            modified_time = datetime.datetime.fromtimestamp(os.path.getmtime(file_path)).strftime("%Y-%m-%d %H:%M:%S")
            size = os.path.getsize(file_path)
            size_str = f"{size:,} bytes"
            
            md_content += f"| {file} | {modified_time} | {size_str} |\n"
    else:
        md_content += "_No HTML files found._\n\n"
    
    # Add other files section
    other_files = [f for f in dir_structure['all_files'] if not f.endswith('.html')]
    md_content += "\n## Other Files\n\n"
    
    if other_files:
        md_content += "| Filename | Type | Last Modified | Size |\n"
        md_content += "|----------|------|--------------|------|\n"
        
        for file in other_files:
            file_path = os.path.join(dir_structure['path'], file)
            file_type = file.split('.')[-1] if '.' in file else "unknown"
            modified_time = datetime.datetime.fromtimestamp(os.path.getmtime(file_path)).strftime("%Y-%m-%d %H:%M:%S")
            size = os.path.getsize(file_path)
            size_str = f"{size:,} bytes"
            
            md_content += f"| {file} | {file_type} | {modified_time} | {size_str} |\n"
    else:
        md_content += "_No other files found._\n\n"
    
    return md_content

# Generate markdown content
markdown_content = create_markdown_listing(directory_structure)

# Save to file
md_file_path = "templates_listing.md"
try:
    with open(md_file_path, 'w') as md_file:
        md_file.write(markdown_content)
    print(f"Successfully wrote directory listing to {md_file_path}")
except Exception as e:
    print(f"Error writing to file: {e}")

# Display the markdown content
print("Preview of generated markdown:\n")
print(markdown_content[:500] + "...\n" if len(markdown_content) > 500 else markdown_content)

## Update base.html

This section will:
1. Read the base.html template file
2. Insert the directory listing information where appropriate
3. Save the updated base.html file

In [ ]:
# Path to the base.html file
base_html_path = os.path.join(template_dir, "base.html")

# Check if base.html exists, if not create a simple template
if not os.path.exists(base_html_path):
    print(f"base.html not found at {base_html_path}. Creating a simple template...")
    
    # Create a basic template
    basic_template = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Default Title{% endblock %}</title>
    <style>
        /* Basic styling */
        body { font-family: Arial, sans-serif; margin: 0; padding: 20px; }
        .container { max-width: 1200px; margin: 0 auto; }
        
        /* Template listing section */
        #template-listing {
            background-color: #f8f9fa;
            padding: 15px;
            margin: 20px 0;
            border-radius: 5px;
        }
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>Application Header</h1>
            <nav>
                <ul>
                    <li><a href="/">Home</a></li>
                    <li><a href="/about">About</a></li>
                </ul>
            </nav>
        </header>
        
        <main>
            <!-- Template listing will be inserted here -->
            <div id="template-listing">
                <h2>Available Templates</h2>
                <!-- TEMPLATE_LISTING_PLACEHOLDER -->
            </div>
            
            {% block content %}
            <p>Default content. This will be replaced by child templates.</p>
            {% endblock %}
        </main>
        
        <footer>
            <p>&copy; 2023 Application Name</p>
        </footer>
    </div>
</body>
</html>
"""
    
    with open(base_html_path, 'w') as base_file:
        base_file.write(basic_template)
    print(f"Created base.html template at {base_html_path}")

# Read the current base.html content
try:
    with open(base_html_path, 'r') as file:
        base_html_content = file.read()
    print("Successfully read base.html")
except Exception as e:
    print(f"Error reading base.html: {e}")
    base_html_content = ""

# Create a backup of the original file
backup_path = base_html_path + ".backup"
try:
    shutil.copy2(base_html_path, backup_path)
    print(f"Created backup of base.html at {backup_path}")
except Exception as e:
    print(f"Error creating backup: {e}")

# Convert markdown listing to HTML for insertion
def markdown_table_to_html(md_content):
    # Simple parser to convert markdown tables to HTML
    html_content = "<div class='template-listing'>\n"
    
    # Process the markdown by sections
    sections = md_content.split('## ')
    
    for section in sections[1:]:  # Skip the first item which is the header
        lines = section.strip().split('\n')
        section_title = lines[0]
        html_content += f"<h3>{section_title}</h3>\n"
        
        if "| Filename |" in section:
            html_content += "<table border='1' class='template-table'>\n"
            
            # Process table headers
            headers = lines[1].strip().split('|')
            headers = [h.strip() for h in headers if h.strip()]
            
            html_content += "<tr>\n"
            for header in headers:
                html_content += f"<th>{header}</th>\n"
            html_content += "</tr>\n"
            
            # Skip the separator row (|----|)
            # Process data rows
            for line in lines[3:]:
                if line.startswith('|'):
                    cells = line.strip().split('|')
                    cells = [c.strip() for c in cells if c.strip()]
                    
                    html_content += "<tr>\n"
                    for cell in cells:
                        html_content += f"<td>{cell}</td>\n"
                    html_content += "</tr>\n"
            
            html_content += "</table>\n"
        else:
            # For non-table content
            html_content += "<p>" + "\n".join(lines[1:]) + "</p>\n"
    
    html_content += "</div>"
    return html_content

# Generate HTML representation of template listing
html_listing = markdown_table_to_html(markdown_content)

# Update base.html with template listing
if "<!-- TEMPLATE_LISTING_PLACEHOLDER -->" in base_html_content:
    updated_content = base_html_content.replace(
        "<!-- TEMPLATE_LISTING_PLACEHOLDER -->", 
        html_listing
    )
    
    # Write the updated content back to base.html
    try:
        with open(base_html_path, 'w') as file:
            file.write(updated_content)
        print(f"Successfully updated base.html with template listing")
    except Exception as e:
        print(f"Error updating base.html: {e}")
else:
    print("Warning: Could not find placeholder for template listing in base.html")
    print("Manual update may be required")

# Display a snippet of the HTML listing
print("\nPreview of HTML listing to be inserted:\n")
print(html_listing[:300] + "...\n" if len(html_listing) > 300 else html_listing)

## Update server.py

This section will:
1. Find and read the server.py file
2. Update it to render or provide access to the template listing
3. Save the updated server.py

In [ ]:
# Define path to server.py
server_py_path = os.path.join("src", "web", "server.py")

# Check if server.py exists, if not create a simple Flask server
if not os.path.exists(server_py_path):
    print(f"server.py not found at {server_py_path}. Creating a simple Flask server...")
    
    # Ensure the directory exists
    os.makedirs(os.path.dirname(server_py_path), exist_ok=True)
    
    # Create a basic Flask server
    basic_server = """from flask import Flask, render_template, jsonify
import os
import datetime

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('base.html', title="Home")

@app.route('/about')
def about():
    return render_template('base.html', title="About")

@app.route('/templates.json')
def templates_json():
    # Get template directory information
    template_dir = os.path.join(os.path.dirname(__file__), "templates")
    template_files = os.listdir(template_dir)
    
    # Create a structured representation
    templates_data = []
    for file in template_files:
        if file.endswith('.html'):
            file_path = os.path.join(template_dir, file)
            templates_data.append({
                'filename': file,
                'modified': datetime.datetime.fromtimestamp(os.path.getmtime(file_path)).isoformat(),
                'size': os.path.getsize(file_path)
            })
    
    return jsonify({
        'templates': templates_data,
        'generated': datetime.datetime.now().isoformat()
    })

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
"""
    
    with open(server_py_path, 'w') as server_file:
        server_file.write(basic_server)
    print(f"Created server.py at {server_py_path}")

# Read the current server.py content
try:
    with open(server_py_path, 'r') as file:
        server_py_content = file.read()
    print("Successfully read server.py")
except Exception as e:
    print(f"Error reading server.py: {e}")
    server_py_content = ""

# Create a backup of the original file
backup_path = server_py_path + ".backup"
try:
    shutil.copy2(server_py_path, backup_path)
    print(f"Created backup of server.py at {backup_path}")
except Exception as e:
    print(f"Error creating backup: {e}")

# Function to check if route already exists in server.py
def route_exists(content, route_path):
    pattern = rf"@app\.route\(['\"]/?{route_path}['\"]"
    return bool(re.search(pattern, content))

# Function to add a new route to server.py
def add_route(content, route_def):
    # Find where to insert the new route
    if_name_pattern = r"if\s+__name__\s*==\s*['\"]__main__['\"]\s*:"
    match = re.search(if_name_pattern, content)
    
    if match:
        # Insert before if __name__ == "__main__"
        insert_pos = match.start()
        updated_content = content[:insert_pos] + route_def + "\n\n" + content[insert_pos:]
        return updated_content
    else:
        # If we can't find the if __name__ block, just append to the end
        return content + "\n\n" + route_def + "\n"

# Check if the templates_listing route exists, if not add it
if not route_exists(server_py_content, "templates_listing"):
    print("Adding templates_listing route to server.py")
    
    new_route = """@app.route('/templates_listing')
def templates_listing():
    \"\"\"Render the templates listing page.\"\"\"
    # Read templates_listing.md if it exists
    try:
        with open('templates_listing.md', 'r') as f:
            md_content = f.read()
    except FileNotFoundError:
        md_content = "Template listing file not found. Please run the directory management notebook."
    
    # You could use a markdown parser here to convert to HTML
    # For simplicity, we'll just pass the raw markdown
    return render_template('base.html', 
                          title="Templates Listing", 
                          markdown_content=md_content)"""
    
    # Add the route to server.py
    server_py_content = add_route(server_py_content, new_route)
    
    # Check if the Flask app import includes render_template
    if "from flask import" in server_py_content and "render_template" not in server_py_content:
        server_py_content = server_py_content.replace(
            "from flask import", 
            "from flask import render_template, "
        )
    
    # Write the updated content back to server.py
    try:
        with open(server_py_path, 'w') as file:
            file.write(server_py_content)
        print("Successfully updated server.py with templates_listing route")
    except Exception as e:
        print(f"Error updating server.py: {e}")
else:
    print("templates_listing route already exists in server.py")

# Display the added route for confirmation
print("\nAdded route to server.py:")
print(new_route)

## Conclusion

This notebook has:

1. Created a directory listing for the templates directory
2. Saved this information to a Markdown file for documentation
3. Updated base.html to include the listing
4. Modified server.py to provide access to the template listing

The generated files can be used as part of your web application to provide a dynamic overview of available templates.

### Next Steps

- Create additional routes to better navigate the templates
- Add sorting and filtering options for the template listing
- Implement an automatic update mechanism to keep the listing current